In [ ]:
# --- Cell 1/5: setup --------------------------------------------------------
REPO_URL = "https://github.com/swaroopms658/loralink-reviewer.git"   # public
%cd /content
!rm -rf /content/loralink && git clone --depth 1 {REPO_URL} /content/loralink
%cd /content/loralink
!pip -q install -r loralink_reviewer_response/requirements-colab.txt

import sys
sys.path.insert(0, "/content/loralink")

# Re-cloning replaces the files on disk, but a kernel that already imported these
# modules keeps the OLD code in sys.modules -- so re-running this cell would
# otherwise leave cell 4 executing the previous clone. Drop them so the next
# import reads the fresh checkout.
_PROJECT = ("main", "cluster_launch", "pipeline_engine", "model_registry",
            "lora_manager", "data_loader", "compression_engine", "benchmarking",
            "device_manager", "network_protocol")
for _name in [m for m in list(sys.modules)
              if m.startswith("loralink_reviewer_response") or m in _PROJECT]:
    del sys.modules[_name]

# Print the commit actually in use -- the fastest way to spot a stale session.
!git -C /content/loralink log --oneline -1
# verify the patched sources against the committed SHA256SUMS (no --update -- a mismatch aborts the run)
!python loralink_reviewer_response/patch/checksums.py --verify

In [ ]:
# --- Cell 2/5: model download ---------------------------------------------
from huggingface_hub import snapshot_download
MODEL = "EleutherAI/gpt-neo-125M"
snapshot_download(MODEL, local_dir=f"./models/{MODEL}",
                  allow_patterns=["*.json", "*.txt", "*.model", "*.safetensors", "*.bin",
                                  "merges.txt", "vocab.json", "tokenizer*"])

In [ ]:
# --- Cell 3/5: params (edit ACCOUNT_TAG and SHARD only) -----------------
ACCOUNT_TAG = "acct1"              # unique per Gmail account
SHARD       = ""  # the only knob besides ACCOUNT_TAG
WALL_BUDGET_MIN = 32
FAILED = 0                         # cells that raised; counted apart from DONE
import time; _NB_START = time.time()
def budget_left(): return WALL_BUDGET_MIN * 60 - (time.time() - _NB_START)

In [ ]:
# --- Cell 4/5: body -- one tiny end-to-end run, prints SMOKE PASS/FAIL --------
import os, pandas as pd
from loralink_reviewer_response.cluster_launch import run_cluster

PER_RUN_ESTIMATE = 120
PLANNED, DONE = 1, 0
csv = f"results_smoke_{ACCOUNT_TAG}.csv"

if budget_left() < PER_RUN_ESTIMATE:
    print("budget exhausted, stopping")
else:
    run_cluster(n_workers=2, dataset="wikitext", seed=0, model=MODEL,
                num_samples=6, epochs=1, tag="smoke", results_csv=csv)
    DONE += 1

if os.path.exists(csv):
    df = pd.read_csv(csv)
    n_loss = int(df["loss"].notna().sum()) if "loss" in df.columns else 0
    print("SMOKE PASS" if n_loss >= 3 else "SMOKE FAIL", f"({n_loss} loss rows)")
else:
    print("SMOKE FAIL (no csv)")


In [ ]:
# --- Cell 5/5: download ---------------------------------------------------
import json, glob, os
from google.colab import files

# main.py writes per-batch rows to results_<kind>_<tag>.csv and the single summary
# row to results_<kind>_<tag>.summary.csv (schemas differ) -- grab both.
produced = sorted(set(glob.glob(f"results_*_{ACCOUNT_TAG}.csv")
                      + glob.glob(f"results_*_{ACCOUNT_TAG}.summary.csv")))

json.dump({"tag": ACCOUNT_TAG, "shard": SHARD,
           "succeeded": DONE, "failed": FAILED, "planned": PLANNED,
           "result_files": [os.path.basename(f) for f in produced],
           "checksums": open("loralink_reviewer_response/patch/SHA256SUMS").read()},
          open(f"run_manifest_{ACCOUNT_TAG}.json", "w"), indent=2)

print(f"succeeded {DONE}/{PLANNED}, failed {FAILED}")
if not produced:
    print("NO RESULT FILES -- every run failed; check cell 4 output above. "
          "Downloading the manifest only.")
else:
    print("result files:", [os.path.basename(f) for f in produced])

for f in produced + [f"run_manifest_{ACCOUNT_TAG}.json"]:
    files.download(f)